# LEAR - backtest

Forecasts are saved under `results/lear/test/full/<model>.csv`.

Backtest of LEAR on the test set, every day of it (364 days). The model is the same as in `03_model_selection/01_LEAR.ipynb`: every day it is recalibrated on the last `calibration_window` days and forecasts all 24 hourly prices of the next day, with one LASSO regression per hour. Two windows are run: 728 and 1092 days, that is 2 and 3 years of 364 days.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent))

from src.config import EPF_EXOG
from src.utils.epf import evaluate_lear, forecast_path
from src.utils.evaluation import compare_models, load_forecasts
from src.utils.file_io import load_final_dataset

/opt/homebrew/Caskroom/miniforge/base/envs/epf-lear-dnn/lib/python3.10/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
dataset_full = load_final_dataset("epf", "pl", "full")
dataset_test = load_final_dataset("epf", "pl", "test")
prices = dataset_full["PL_price"]

configs = [
    {"model": "LEAR_728", "calibration_window": 728, "exog": EPF_EXOG},
    {"model": "LEAR_1092", "calibration_window": 1092, "exog": EPF_EXOG},
]

test_start = dataset_test.index.min()
test_end = dataset_test.index.max()

## 1. Backtest

Each window is run once over every day of the test set. The forecasts are saved to `results/lear/test/full/<model>.csv`, and the comparison below reads the files instead of recomputing them.

In [ ]:
for config in configs:
    evaluate_lear(
        dataset=dataset_full,
        calibration_window=config["calibration_window"],
        exog=config["exog"],
        forecast_start=test_start,
        forecast_end=test_end,
        path=forecast_path(config["model"], split="test", run="full"),
    )

## 2. Backtest comparison

Forecasts are scored against `PL_price`. MAE and RMSE are in EUR/MWh. rMAE is the MAE divided by the MAE of a naive forecast that repeats the price from the same hour one week earlier, so a value below 1 beats that naive forecast. Lower is better for all three.

In [4]:
models = [config["model"] for config in configs]
forecasts = load_forecasts(
    [forecast_path(model, split="test", run="full") for model in models]
)

compare_models(forecasts, prices, metrics=["rMAE", "MAE", "RMSE", "sMAPE"])

,rMAE,MAE,RMSE,sMAPE
model,,,,
LEAR_728,0.603856,22.395072,36.400590,0.282485
LEAR_1092,0.598969,22.213831,35.660759,0.280358
